# Foot Alignment Full Pipeline

This notebook records the reproducible steps used for the clean foot-aware shoe alignment pipeline.

The final foot-aware alignment outputs are written under:

```text
/data/abelde/projects/active/Shell_Gaussian/FootShellGaussian/output/foot_aware_alignment
```


## Section 1: Turntable Canonicalization

The original processed dataset is here:

```text
/data/abelde/datasets/processed/gshell_shoes
```

The images look like a consistent turntable dataset, but COLMAP can choose a different global yaw for each shoe. That means two shoes can have the same image ordering, but their trained meshes can face different world directions.

This step fixes that by creating the processed dataset used for the current GShell training:

```text
/data/abelde/datasets/processed/gshell_shoes_turntable_canonical
```

What changes:

- `transforms.json` is rewritten.
- Each frame's `transform_matrix` is yaw-rotated.
- `img01.jpg` is forced to land at the same turntable angle for every shoe.

What does not change:

- The actual image files are not rotated.
- The mask files are not changed.
- `camera_angle_x` is not changed.
- `file_path` is not changed.
- Physical scale is not recovered.

So this is not image editing and not physical-size calibration. It is camera-pose phase correction. The output dataset keeps image and mask folders as symlinks to the original processed dataset, while writing new `transforms.json` files.


In [ ]:
import json
import runpy
import shlex
import sys
from pathlib import Path

PROJECT_ROOT = Path(globals().get('PROJECT_ROOT', '/data/abelde/projects/active/Shell_Gaussian'))

# Set this to True when you want to regenerate the dataset.
RUN_TURNTABLE_CANONICALIZATION = False

FOOTSHELL_ROOT = PROJECT_ROOT / 'FootShellGaussian'
GSHELL_ENV = PROJECT_ROOT / 'baselines' / 'GShell' / 'GShell_env'

CANON_INPUT_ROOT = Path('/data/abelde/datasets/processed/gshell_shoes')
CANON_OUTPUT_ROOT = Path('/data/abelde/datasets/processed/gshell_shoes_turntable_canonical')

# Leave this empty to canonicalize every shoe in CANON_INPUT_ROOT.
# Add exact scene folder names if you only want a subset.
CANON_SCENE_NAMES = []

CANON_REFERENCE_FRAME = 'img01.jpg'
CANON_TARGET_ANGLE_DEG = 90
CANON_OVERWRITE = True
CANON_DRY_RUN = False

canonicalize_script = FOOTSHELL_ROOT / 'dataset' / 'canonicalize_gshell_turntable_phase.py'
canonicalize_argv = [
    str(canonicalize_script),
    '--input-root', str(CANON_INPUT_ROOT),
    '--output-root', str(CANON_OUTPUT_ROOT),
    '--reference-frame', CANON_REFERENCE_FRAME,
    '--target-angle-deg', str(CANON_TARGET_ANGLE_DEG),
]
for scene_name in CANON_SCENE_NAMES:
    canonicalize_argv += ['--scene', scene_name]
if CANON_OVERWRITE:
    canonicalize_argv += ['--overwrite']
if CANON_DRY_RUN:
    canonicalize_argv += ['--dry-run']

shell_equivalent = [str(GSHELL_ENV / 'bin' / 'python')] + canonicalize_argv
print(' '.join(shlex.quote(part) for part in shell_equivalent))
print('canonical output root:', CANON_OUTPUT_ROOT)

if RUN_TURNTABLE_CANONICALIZATION:
    old_argv = sys.argv[:]
    try:
        sys.argv = canonicalize_argv
        runpy.run_path(str(canonicalize_script), run_name='__main__')
    finally:
        sys.argv = old_argv
else:
    print('Set RUN_TURNTABLE_CANONICALIZATION = True, then execute this cell to run Section 1.')

summary_path = CANON_OUTPUT_ROOT / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    rows = summary.get('rows', summary if isinstance(summary, list) else [])
    print('summary:', summary_path)
    print('row_count:', len(rows))
    if isinstance(summary, dict):
        for key in [
            'scene_count',
            'status_counts',
            'max_target_error_deg',
            'all_rotations_passed',
            'all_frame_counts_36',
            'original_dataset_modified',
        ]:
            if key in summary:
                print(f'{key}:', summary[key])


## Section 2: Train GShell Meshes

This step trains GShell using the turntable-canonical dataset from Section 1.

Input dataset:

```text
/data/abelde/datasets/processed/gshell_shoes_turntable_canonical
```

Output meshes:

```text
/data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/turntable-512-768
```

The important training settings are:

- config: `shoes_mc_normfix_512_768.json`
- output suffix: `_turntable`
- `SKIP_EXISTING=1`, so already completed shoes are skipped

This cell launches the existing GShell tmux training script. It does not run by default because this is a long GPU job.


In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(globals().get('PROJECT_ROOT', '/data/abelde/projects/active/Shell_Gaussian'))
FOOTSHELL_ROOT = PROJECT_ROOT / 'FootShellGaussian'
GSHELL_ROOT = PROJECT_ROOT / 'baselines' / 'GShell'
GSHELL_OUTPUT_ROOT = GSHELL_ROOT / 'output'

# Safety guard: set this to True when you want to launch the tmux training job.
RUN_GSHELL_TRAINING = False

TRAIN_SHOE_NAMES = [
    'Adidas-Yeezy-Boost-350-V2-Desert-Sage-Infant',
    'Adidas-Yeezy-Boost-350-V2-Static-Non-Reflective-Infants',
    'Adidas-Yeezy-Boost-350-V2-Static-Non-Reflective-Kids',
    'Air-Jordan-1-Mid-Wear-Away-Chicago-Gs',
    'Air-Jordan-1-Retro-High-Hyper-Royal-Smoke-Grey-Gs',
    'Air-Jordan-1-Retro-High-Og-Washed-Black-Gs',
    'Air-Jordan-1-Retro-High-Og-White-Cement-Gs',
    'Air-Jordan-12-Retro-Arctic-Punch-Gs',
    'Air-Jordan-13-Retro-Houndstooth-Gs',
    'Air-Jordan-5-Retro-Plaid-Gs',
    'Air-Jordan-6-Retro-Washed-Denim-2022-Gs',
    'Birkenstock-Boston-Suede-Stone-Coin',
    'Crocs-Classic-Clog-Cinnamon-Toast-Crunch',
    'Crocs-Classic-Clog-Cinnamon-Toast-Crunch-Gs',
    'Crocs-Classic-Clog-Cocoa-Puffs-Kids',
    'Crocs-Classic-Clog-Staple-Sidewalk-Luxe',
    'Nike-Calm-Slide-Cinnamon-Monarch',
    'Nike-Cortez-Se-Suede-Pacific-Moss-Infinite-Gold-Muslin-Sail',
    'Ugg-Bailey-Bow-Ii-Boot-Ribbon-Red-Kids',
    'Ugg-Classic-Short-Ii-Boot-Rock-Rose-Toddler',
]

# To train every scene in the dataset, set TRAIN_SHOE_NAMES = [].
TRAIN_SESSION_NAME = 'gshell_turntable_20'
TRAIN_DATASET_ROOT = Path('/data/abelde/datasets/processed/gshell_shoes_turntable_canonical')
TRAIN_CONFIG = GSHELL_ROOT / 'configs' / 'shoes_mc_normfix_512_768.json'
TRAIN_OUTPUT_ROOT = GSHELL_OUTPUT_ROOT / 'turntable-512-768'
TRAIN_SCRIPT = GSHELL_ROOT / 'scripts' / 'train_all_shoes_tmux.sh'

train_env = os.environ.copy()
train_env.update({
    'MIN_FREE_MB': '51200',
    'MAX_PARALLEL_JOBS': '5',
    'SKIP_EXISTING': '1',
    'GSHELL_DATASET_ROOT': str(TRAIN_DATASET_ROOT),
    'GSHELL_CONFIG': str(TRAIN_CONFIG),
    'GSHELL_OUT_SUFFIX': '_turntable',
    'GSHELL_OUTPUT_ROOT': str(TRAIN_OUTPUT_ROOT),
})
train_cmd = ['bash', str(TRAIN_SCRIPT), TRAIN_SESSION_NAME] + TRAIN_SHOE_NAMES

env_prefix = ' '.join(f'{key}={shlex.quote(train_env[key])}' for key in [
    'MIN_FREE_MB', 'MAX_PARALLEL_JOBS', 'SKIP_EXISTING', 'GSHELL_DATASET_ROOT',
    'GSHELL_CONFIG', 'GSHELL_OUT_SUFFIX', 'GSHELL_OUTPUT_ROOT'
])
print('cd', GSHELL_ROOT)
print(env_prefix + ' ' + ' '.join(shlex.quote(part) for part in train_cmd))
print('tmux attach command:', f'tmux attach -t {TRAIN_SESSION_NAME}')

if RUN_GSHELL_TRAINING:
    subprocess.run(train_cmd, cwd=str(GSHELL_ROOT), env=train_env, check=True)
else:
    print('Set RUN_GSHELL_TRAINING = True, then execute this cell to launch training.')

mesh_count = len(list(TRAIN_OUTPUT_ROOT.glob('*/mesh/mesh.obj')))
watertight_count = len(list(TRAIN_OUTPUT_ROOT.glob('*/mesh_watertight/mesh.obj')))
print('existing open mesh count:', mesh_count)
print('existing watertight mesh count:', watertight_count)


## Section 3: Run Clean Foot-Aware Alignment

This is the final one-shot foot placement step. For each trained GShell shoe mesh, it runs:

```text
trained shoe mesh
-> support footprint + pseudo-footbed extraction
-> initial SUPR foot alignment with length_ratio=0.85
-> optimized foot fitting
-> diagnostics and summaries
```

Input trained meshes:

```text
/data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/turntable-512-768
```

Final outputs:

```text
/data/abelde/projects/active/Shell_Gaussian/FootShellGaussian/output/foot_aware_alignment
```


In [1]:
import importlib
import os
import runpy
import shlex
import sys
from pathlib import Path

PROJECT_ROOT = Path(globals().get('PROJECT_ROOT', '/data/abelde/projects/active/Shell_Gaussian'))
FOOTSHELL_ROOT = PROJECT_ROOT / 'FootShellGaussian'
GSHELL_ROOT = PROJECT_ROOT / 'baselines' / 'GShell'
GSHELL_ENV = GSHELL_ROOT / 'GShell_env'
GSHELL_OUTPUT_ROOT = GSHELL_ROOT / 'output'

# Safety guard: set this to True when you want to run the final foot-aware pipeline.
RUN_FOOT_AWARE_ALIGNMENT = True

# Debug-friendly execution. This uses runpy so notebook breakpoints can work.
# It also clears cached FootShellGaussian modules first, so recently edited files are re-imported.
RELOAD_FOOTSHELL_MODULES = True

FOOT_AWARE_CUDA_VISIBLE_DEVICES = '5'
FOOT_AWARE_SHOE_NAMES = []
# Smoke-test example:
# FOOT_AWARE_SHOE_NAMES = [
#     'Adidas-Yeezy-Boost-350-V2-Desert-Sage-Infant',
#     'Birkenstock-Boston-Suede-Stone-Coin',
#     'Crocs-Classic-Clog-Cinnamon-Toast-Crunch-Gs',
#     'Ugg-Classic-Short-Ii-Boot-Rock-Rose-Toddler',
# ]
FOOT_AWARE_OVERWRITE = True

FOOT_AWARE_MESH_ROOT = GSHELL_OUTPUT_ROOT / 'turntable-512-768'
FOOT_AWARE_OUTPUT_ROOT = FOOTSHELL_ROOT / 'output' / 'foot_aware_alignment'
FOOT_AWARE_FOOT_OBJ = PROJECT_ROOT / 'baselines' / 'SUPR' / 'output' / 'debug_playground' / 'supr_male_right_foot_neutral.obj'
FOOT_AWARE_SCRIPT = FOOTSHELL_ROOT / 'scripts' / 'run_foot_aware_alignment_pipeline.py'

foot_aware_argv = [
    str(FOOT_AWARE_SCRIPT),
    '--mesh-root', str(FOOT_AWARE_MESH_ROOT),
    '--output-root', str(FOOT_AWARE_OUTPUT_ROOT),
    '--foot-obj', str(FOOT_AWARE_FOOT_OBJ),
    '--device', 'cuda',
    '--style-mode', 'auto',
    '--length-ratio', '0.85',
    '--diagnostics', 'minimal',
    '--adam-steps', '160',
    '--lbfgs-steps', '25',
]
for shoe_name in FOOT_AWARE_SHOE_NAMES:
    foot_aware_argv += ['--shoe-name', shoe_name]
if FOOT_AWARE_OVERWRITE:
    foot_aware_argv += ['--overwrite']

shell_equivalent = [str(GSHELL_ENV / 'bin' / 'python')] + foot_aware_argv
print('CUDA_VISIBLE_DEVICES=' + FOOT_AWARE_CUDA_VISIBLE_DEVICES + ' ' + ' '.join(shlex.quote(part) for part in shell_equivalent))

def clear_footshell_import_cache():
    module_names = list(sys.modules)
    for module_name in module_names:
        if module_name == 'FootShellGaussian' or module_name.startswith('FootShellGaussian.'):
            del sys.modules[module_name]
        elif module_name in {'run_foot_fit_optimization', 'run_foot_aware_alignment_pipeline'}:
            del sys.modules[module_name]
    importlib.invalidate_caches()

if RUN_FOOT_AWARE_ALIGNMENT:
    old_argv = sys.argv[:]
    old_cuda = os.environ.get('CUDA_VISIBLE_DEVICES')
    os.environ['CUDA_VISIBLE_DEVICES'] = FOOT_AWARE_CUDA_VISIBLE_DEVICES
    try:
        if RELOAD_FOOTSHELL_MODULES:
            clear_footshell_import_cache()
        sys.argv = foot_aware_argv
        runpy.run_path(str(FOOT_AWARE_SCRIPT), run_name='__main__')
    finally:
        sys.argv = old_argv
        if old_cuda is None:
            os.environ.pop('CUDA_VISIBLE_DEVICES', None)
        else:
            os.environ['CUDA_VISIBLE_DEVICES'] = old_cuda


CUDA_VISIBLE_DEVICES=5 /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env/bin/python /data/abelde/projects/active/Shell_Gaussian/FootShellGaussian/scripts/run_foot_aware_alignment_pipeline.py --mesh-root /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/turntable-512-768 --output-root /data/abelde/projects/active/Shell_Gaussian/FootShellGaussian/output/foot_aware_alignment --foot-obj /data/abelde/projects/active/Shell_Gaussian/baselines/SUPR/output/debug_playground/supr_male_right_foot_neutral.obj --device cuda --style-mode auto --length-ratio 0.85 --diagnostics minimal --adam-steps 160 --lbfgs-steps 25 --overwrite
[1/20] Adidas-Yeezy-Boost-350-V2-Desert-Sage-Infant_turntable


KeyboardInterrupt: 

## Section 4: Build SUPR-Derived Pseudo-Last

This step uses the foot-aware alignment artifacts from Section 3 to build a smooth pseudo-last mesh. It consumes the aligned SUPR foot, support footprint JSON, and pseudo-footbed heightmap, then writes a watertight last mesh and SDF under:

```text
/data/abelde/projects/active/Shell_Gaussian/FootShellGaussian/output/pseudo_last
```

This is a post-alignment geometry prior stage. It does not modify GShell training or mSDF logic yet.


In [ ]:
import importlib
import runpy
import shlex
import sys
from pathlib import Path

PROJECT_ROOT = Path(globals().get('PROJECT_ROOT', '/data/abelde/projects/active/Shell_Gaussian'))
FOOTSHELL_ROOT = PROJECT_ROOT / 'FootShellGaussian'
GSHELL_ROOT = PROJECT_ROOT / 'baselines' / 'GShell'
GSHELL_ENV_PYTHON = GSHELL_ROOT / 'GShell_env' / 'bin' / 'python'

# Set this to True when you want to build pseudo-lasts.
RUN_PSEUDO_LAST_BUILDER = False

ALIGNMENT_ROOT = FOOTSHELL_ROOT / 'output' / 'foot_aware_alignment'
PSEUDO_LAST_OUTPUT_ROOT = FOOTSHELL_ROOT / 'output' / 'pseudo_last'
PSEUDO_LAST_SCRIPT = FOOTSHELL_ROOT / 'scripts' / 'run_pseudo_last_builder.py'

# Leave empty to process every normal closed shoe in ALIGNMENT_ROOT.
PSEUDO_LAST_SHOES = [
    'Adidas-Yeezy-Boost-350-V2-Desert-Sage-Infant',
]

BUILD_PSEUDO_LAST_SDF = True
ALLOW_NON_NORMAL_PSEUDO_LAST = False

argv = [
    str(PSEUDO_LAST_SCRIPT),
    '--alignment-root', str(ALIGNMENT_ROOT),
    '--output-root', str(PSEUDO_LAST_OUTPUT_ROOT),
    '--device', 'cuda',
    '--sdf-resolution', '128',
    '--overwrite',
]
for shoe_name in PSEUDO_LAST_SHOES:
    argv.extend(['--shoe-name', shoe_name])
if not BUILD_PSEUDO_LAST_SDF:
    argv.append('--no-sdf')
if ALLOW_NON_NORMAL_PSEUDO_LAST:
    argv.append('--allow-non-normal')

print('Equivalent shell command:')
print(shlex.join([str(GSHELL_ENV_PYTHON)] + argv))

if RUN_PSEUDO_LAST_BUILDER:
    sys.argv = argv
    importlib.invalidate_caches()
    runpy.run_path(str(PSEUDO_LAST_SCRIPT), run_name='__main__')
else:
    print('Set RUN_PSEUDO_LAST_BUILDER = True to execute this cell.')
